In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

пробуем -тестируем создание нейросети по курсу дубинина сергея с рутуба

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# когда используется Sequential, нет необходимости расписывать класс нейросети, методы инит и форвард, все это уже зашито
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

In [ ]:
model

модель создана, теперь ее нужно проверить

In [ ]:
input = torch.rand([16,784], dtype=torch.float32)

out = model(input)
out.shape

рассмотрим полезные методы, которыми будем пользоваться далее

In [ ]:
# model.state_dict()
# model.state_dict()['0.weight']
# model.state_dict()['2.bias']
# for parameter in model.parameters():
#     print(parameter)
#     print(parameter.shape, end='\n\n')

# model.train()
# model.val()

второй способ создания последовательной модели с помощью класа sequential

In [ ]:
model = nn.Sequential()
model.add_module('layer_1', nn.Linear(784, 128))
model.add_module('relu', nn.ReLU())
model.add_module('layer_2', nn.Linear(128, 10))

In [ ]:
# model
# model.layer_1
model.relu

In [ ]:
input = torch.rand([16,784], dtype=torch.float32)

out = model(input)
out.shape

In [ ]:
model.state_dict()

теперь создадим модель нейронной сети на основе собственного класса

In [ ]:
class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.act(x)
        out = self.layer_2(x)
        return out


model = MyModel(784, 10)

In [ ]:
input = torch.rand([16,784], dtype=torch.float32)

out = model(input)
out.shape

теперь создадим модель с двумя входами и двумя выходами

In [ ]:
class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.act = nn.ReLU()

    def forward(self, x, y):
        x = self.layer_1(x)
        x = self.act(x+y)
        out = self.layer_2(x)
        return out, x


model = MyModel(784, 10)

In [ ]:
x = torch.rand([16,784], dtype=torch.float32)
y = torch.rand([16,128], dtype=torch.float32)

out = model(x,y)
len(out)

In [ ]:
print(f"out_shape_1 = {out[0].shape}")
print(f"out_shape_2 = {out[1].shape}")

создание модели при помощи класса ModuleList 

In [ ]:
class MyModel(nn.Module):
    def __init__(self, input, output, hidden_size=2048):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(10):
            self.layers.add_module(f'layer_{i}', nn.Linear(input, hidden_size))
            self.layers.add_module(f'act_{i}', nn.ReLU())
            input = hidden_size
            hidden_size = int(hidden_size/2)
        self.layers.add_module('foutput_{i}', nn.Linear(input, output))

    def forward(self, x):
        output = []
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i != 0 and i%2 == 0 and i%4 != 0:
                output.append(x)
                # print(len(x))
        output.append(x)
        return output


model = MyModel(784, 10)

In [ ]:
model

In [ ]:
input = torch.rand([16,784], dtype=torch.float32)

out = model(input)


In [ ]:
# len(out)
print(f"out_shape_1 = {out[2].shape}")

копируем ту же модель, но добавляем использование ModuleDict

In [ ]:
class MyModel(nn.Module):
    def __init__(self, input, output, hidden_size=2048, choice='relu'):
        super().__init__()
        self.activations = nn.ModuleDict({
            'lrelu': nn.LeakyReLU(),
            'relu': nn.ReLU()
        })
        self.layers = nn.ModuleList()
        for i in range(10):
            self.layers.add_module(f'layer_{i}', nn.Linear(input, hidden_size))
            self.layers.add_module(f'act_{i}', self.activations[choice])
            input = hidden_size
            hidden_size = int(hidden_size/2)
        self.layers.add_module('foutput_{i}', nn.Linear(input, output))

    def forward(self, x):
        output = []
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i != 0 and i%2 == 0 and i%4 != 0:
                output.append(x)
                # print(len(x))
        output.append(x)
        return output

In [ ]:
model = MyModel(784, 10, choice='lrelu')
model

создание нейросети для классификации MNIST

In [ ]:
class MyModel(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input, 128)
        self.layer_2 = nn.Linear(128, output)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.layer_1(x)
        x = self.act(x)
        out = self.layer_2(x)
        return out


In [ ]:
model_classification = MyModel(784, 10)

In [ ]:
# определяем функцию потерь и оптимизатор градиентного спуска
loss_classiification = nn.CrossEntropyLoss()
opt_classiification = torch.optim.Adam(model_classification.parameters(), lr=0.001)

In [ ]:
input = torch.rand([16,784], dtype=torch.float32)

out = model_classification(input)
out.shape

создадим модель для задачи регрессии

In [ ]:
model_regression = MyModel(64*64, 2)

In [ ]:
# определяем функцию потерь и оптимизатор градиентного спуска
loss_regression = nn.MSELoss()
opt_regression = torch.optim.Adam(model_classification.parameters(), lr=0.001)

In [ ]:
input = torch.rand([16,64*64], dtype=torch.float32)

out = model_regression(input)
out.shape

In [ ]:
import torch
print(f"PyTorch версия: {torch.__version__}")
print(f"Доступен CUDA: {torch.cuda.is_available()}")

# Создайте тензор
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x**2
loss = y.sum()
loss.backward()
print(f"Градиент: {x.grad}")  # Должно быть: [2.0, 4.0, 6.0]



print(f"Тензор: {z1}")
print(f"Сумма по строкам: {sum_rows}")
print(f"Сумма по колонкам: {sum_cols}")

In [ ]:
z=torch.ones(3,3)
z1=torch.randn(3,3)
y1 = z1 * 2
sum_cols = y1.sum(dim=0)
sum_rows = y1.sum(dim=1)

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = torch.tensor([-4.0], requires_grad=True)

f = (x + y) ** 2 + 2 * x * y

f.backward()



print(f)

print(x.data, x.grad)
print(y.data, y.grad)